In [1]:
import argparse
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils.subgraph import k_hop_subgraph

from feat_func import data_process
from models import GEARSage
from utils import DGraphFin
from utils.evaluator import Evaluator
from utils.utils import prepare_folder


def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train(model, data, optimizer):
    model.train()

    optimizer.zero_grad()
    neg_idx = data.train_neg[
        torch.randperm(data.train_neg.size(0))[: data.train_pos.size(0)]
    ]
    train_idx = torch.cat([data.train_pos, neg_idx], dim=0)

    nodeandneighbor, edge_index, node_map, mask = k_hop_subgraph(
        train_idx, 3, data.edge_index, relabel_nodes=True, num_nodes=data.x.size(0)
    )

    out = model(
        data.x[nodeandneighbor],
        edge_index,
        data.edge_attr[mask],
        data.edge_timestamp[mask],
        data.edge_direct[mask],
    )
    loss = F.nll_loss(out[node_map], data.y[train_idx])
    loss.backward()

    nn.utils.clip_grad_norm_(model.parameters(), 2.0)

    optimizer.step()
    torch.cuda.empty_cache()
    return loss.item()


@torch.no_grad()
def test(model, data):
    model.eval()
    out = model(
        data.x, data.edge_index, data.edge_attr, data.edge_timestamp, data.edge_direct,
    )

    y_pred = out.exp()
    return y_pred


def main():
    parser = argparse.ArgumentParser(description="GEARSage for DGraphFin Dataset")
    parser.add_argument("--dataset", type=str, default="DGraphFin")
    parser.add_argument("--model", type=str, default="GEARSage")
    parser.add_argument("--device", type=int, default=0)
    parser.add_argument("--epochs", type=int, default=300)
    parser.add_argument("--hiddens", type=int, default=128)
    parser.add_argument("--layers", type=int, default=3)
    parser.add_argument("--dropout", type=float, default=0.30)

    args = parser.parse_args()
    print(args)

    device = f"cuda:{args.device}" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)
    model_dir = prepare_folder(args.dataset, args.model)
    print("model_dir:", model_dir)
    set_seed(42)
    dataset = DGraphFin(root="./dataset", name="DGraphFin")

    nlabels = 2

    data = dataset[0]

    split_idx = {
        "train": data.train_mask,
        "valid": data.valid_mask,
        "test": data.test_mask,
    }

    data = data_process(data).to(device)
    train_idx = split_idx["train"].to(device)

    data.train_pos = train_idx[data.y[train_idx] == 1]
    data.train_neg = train_idx[data.y[train_idx] == 0]
    model = GEARSage(
        in_channels=data.x.size(-1),
        hidden_channels=args.hiddens,
        out_channels=nlabels,
        num_layers=args.layers,
        dropout=args.dropout,
        activation="elu",
        bn=True,
    ).to(device)

    print(f"Model {args.model} initialized")
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.0)
    best_auc = 0.0
    evaluator = Evaluator("auc")
    y_train, y_valid = data.y[data.train_mask], data.y[data.valid_mask]
    for epoch in range(1, args.epochs + 1):
        loss = train(model, data, optimizer)
        out = test(model, data)
        preds_train, preds_valid = out[data.train_mask], out[data.valid_mask]
        preds_train = preds_train.squeeze()
        preds_valid = preds_valid.squeeze()
        train_auc = evaluator.eval(y_train, preds_train)["auc"]
        valid_auc = evaluator.eval(y_valid, preds_valid)["auc"]

        if valid_auc >= best_auc:
            best_auc = valid_auc
            torch.save(model.state_dict(), model_dir + "model.pt")
            # 保存测试集的预测标签
            with open(model_dir + "preds_test.txt", "w") as f:
                for item in preds_test.tolist():
                    f.write("%s\n" % item)
            # 保存训练集和验证集的预测标签
            with open(model_dir + "preds_train.txt", "w") as f:
                for item in preds_train.tolist():
                    f.write("%s\n" % item)
            with open(model_dir + "preds_valid.txt", "w") as f:
                for item in preds_valid.tolist():
                    f.write("%s\n" % item)
        print(
            f"Epoch: {epoch:02d}, "
            f"Loss: {loss:.4f}, "
            f"Train: {train_auc:.2%}, "
            f"Valid: {valid_auc:.2%},"
            f"Best: {best_auc:.4%},"
        )
        torch.cuda.empty_cache()  # 清理缓存

    test_auc = evaluator.eval(data.y[data.test_mask], preds)["auc"]
    print(f"test_auc: {test_auc}")


if __name__ == "__main__":
    main()

/root/miniconda3/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
usage: ipykernel_launcher.py [-h] [--dataset DATASET] [--model MODEL]
                             [--device DEVICE] [--epochs EPOCHS]
                             [--hiddens HIDDENS] [--layers LAYERS]
                             [--dropout DROPOUT]
ipykernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-cff4cadc-3861-49f2-9d06-5fff512f6d7d.json


SystemExit: 2

/root/miniconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
